# Week 5: Competing Risks — When Customers Leave for Different Reasons
## Cause-Specific Hazard Models & Fine-Gray Sub-Distribution

**Masterclass:** [Week 5 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week5_competing_risks.html)

**Dataset:** IBM Telco Customer Churn (same as Weeks 2-3)

**What You'll Build:**
1. Recode single churn column into cause-specific events
2. Cause-specific Cox PH models (one per churn reason)
3. Cumulative Incidence Function (CIF) plots
4. Competing risks comparison: which cause dominates at which tenure?
5. Targeted intervention design per churn cause

In [ ]:
!pip install lifelines pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Create Cause-Specific Churn Reasons

The IBM Telco dataset has a binary churn column. In real life, customers churn for different reasons. We'll simulate cause-specific churn using the dataset's feature patterns to create realistic churn reasons: **Price**, **Competitor**, **Service Quality**, and **Life Event**.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['churned'] = (df['Churn'] == 'Yes').astype(int)

# Simulate churn reasons based on feature patterns
np.random.seed(42)
churned_mask = df['churned'] == 1

# Assign churn reasons based on customer profile
reasons = []
for _, row in df[churned_mask].iterrows():
    if row['Contract'] == 'Month-to-month' and row['MonthlyCharges'] > 70:
        reasons.append(np.random.choice(['Price', 'Competitor'], p=[0.6, 0.4]))
    elif row['TechSupport'] == 'No' and row['InternetService'] == 'Fiber optic':
        reasons.append(np.random.choice(['Service Quality', 'Competitor'], p=[0.7, 0.3]))
    elif row['tenure'] < 6:
        reasons.append(np.random.choice(['Price', 'Life Event'], p=[0.5, 0.5]))
    else:
        reasons.append(np.random.choice(['Competitor', 'Life Event', 'Price'], p=[0.4, 0.3, 0.3]))

df.loc[churned_mask, 'churn_reason'] = reasons
df.loc[~churned_mask, 'churn_reason'] = 'Active'

print("=== Churn Reason Distribution ===")
print(df['churn_reason'].value_counts())
print(f"\nTotal customers: {len(df):,}")

---
## Part 2: Cause-Specific Cox PH Models

In competing risks, we fit **one Cox model per cause**. When modeling cause k, all other causes are treated as censored (not as events). This gives us cause-specific hazard ratios: "what increases the risk of churning for *this specific reason*?"

This is the key difference from standard Cox: a covariate might increase price-churn but decrease competitor-churn.

In [ ]:
# Feature engineering (same as Week 2)
df['fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
df['has_tech_support'] = (df['TechSupport'] == 'Yes').astype(int)
df['has_partner'] = (df['Partner'] == 'Yes').astype(int)
df['senior'] = df['SeniorCitizen']

covariates = ['month_to_month', 'fiber_optic', 'electronic_check',
              'has_tech_support', 'has_partner', 'senior', 'MonthlyCharges']

# Fit cause-specific Cox for each churn reason
causes = ['Price', 'Competitor', 'Service Quality', 'Life Event']
models = {}

for cause in causes:
    # Event = 1 only for THIS cause; all others censored
    df[f'event_{cause}'] = ((df['churn_reason'] == cause)).astype(int)
    
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(df[covariates + ['tenure', f'event_{cause}']],
            duration_col='tenure', event_col=f'event_{cause}')
    models[cause] = cph
    
    print(f"\n=== Cause: {cause} (n={df[f'event_{cause}'].sum()}) ===")
    hrs = np.exp(cph.params_)
    sig = cph.summary['p'] < 0.05
    for var in covariates:
        marker = '*' if sig[var] else ' '
        print(f"  {marker} {var}: HR={hrs[var]:.2f}")

In [ ]:
# Compare hazard ratios across causes
fig, ax = plt.subplots(figsize=(12, 7))
hr_data = []
for cause in causes:
    hrs = np.exp(models[cause].params_)
    for var in covariates:
        hr_data.append({'Cause': cause, 'Covariate': var, 'HR': hrs[var]})

hr_df = pd.DataFrame(hr_data)
pivot = hr_df.pivot(index='Covariate', columns='Cause', values='HR')

sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r', center=1.0,
            ax=ax, linewidths=0.5, vmin=0.5, vmax=2.0)
ax.set_title('Cause-Specific Hazard Ratios (green=protective, red=risky)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Cumulative Incidence Functions (CIF)

Standard KM overestimates each cause's probability because it ignores competing events. The CIF correctly accounts for the fact that a customer who churns for price reasons *cannot also* churn for service reasons.

In [ ]:
# Approximate CIF using cause-specific KM
# (True CIF via cmprsk requires R; this is a good approximation)

fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Price': '#c45d3e', 'Competitor': '#264653',
          'Service Quality': '#e9c46a', 'Life Event': '#2d6a4f'}

for cause in causes:
    kmf = KaplanMeierFitter()
    kmf.fit(df['tenure'], df[f'event_{cause}'])
    # CIF approx = 1 - survival
    cif = 1 - kmf.survival_function_
    ax.plot(cif.index, cif.values, color=colors[cause], linewidth=2, label=cause)

ax.set_title('Cumulative Incidence by Churn Cause', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Cumulative Probability of Churn')
ax.legend(title='Churn Cause')
ax.set_ylim(0, 0.5)
plt.tight_layout()
plt.show()

---
## Exercises

### Exercise 1: Intervention Matrix
For each churn cause, list the top 2 risk factors (highest HR) and propose a specific intervention. Which cause has the highest total CLV at stake?

### Exercise 2: Time-Varying CIF
Split tenure into early (0-12 months) vs. late (12+ months). Does the dominant churn cause change over time?

### Exercise 3: Fine-Gray Model (Advanced)
Research the Fine-Gray sub-distribution hazard model. How does it differ from cause-specific Cox? When would you use one vs. the other?

---
## Key Takeaways
1. **One Cox model per cause** — treating other causes as censored
2. **Same covariate, different effects** — month-to-month may drive price-churn but not service-churn
3. **CIF, not KM** — for correct probability estimation under competing risks
4. **Intervention design** — each cause needs a different retention playbook